# Alternative Search A/B Core Dataset Schema and Generation Context

## Overview

This dataset represents **search request events** collected during an A/B test of a commerce retrieval system. Each row is one search-related event observed for a user, session, and request. The main comparison is between a **Baseline** retrieval pipeline and an **Alternate** retrieval pipeline that augments standard retrieval with an **alternative search** component.

The dataset combines:
- **experiment assignment metadata**
- **request and session identifiers**
- **user-entered query text**
- **internally normalized query text**
- **retrieval and response characteristics**
- **engagement outcomes** such as clicks and orders
- **mechanism labels** specific to the Alternate flow
- **frontend trigger context**

This document explains each field, the physical process that generates it, and the business meaning behind it so the dataset can be understood and recreated for testing.

---

## field mapping

|  concept | field name |
|---|---|
| Experiment arm | `COHORT_ID` |
| Client identifier | `ACTOR_ID` |
| Visit/session identifier | `SESSION_KEY` |
| Search identifier | `REQUEST_KEY` |
| Dedup / event identifier | `EVENT_KEY` |
| Request start time | `REQUEST_TS` |
| Response time | `LATENCY_MS` |
| Result count | `ITEM_COUNT` |
| Raw query | `INPUT_TEXT` |
| Clicked ranks | `ENGAGED_POSITIONS` |
| Purchased ranks | `ORDERED_POSITIONS` |
| Revenue | `ORDER_VALUE` |
| Variant mechanism status | `ALT_FLOW_STATUS` |
| Final processed query | `NORMALIZED_TEXT` |
| Trigger / action cause | `TRIGGER_MODE` |


---

## Physical process that generates a row

A realistic request row is produced by the following pipeline:

1. **A user interacts with the search box**
   - The user may type in the box without explicitly submitting the search.
   - Some websites call the search backend during typing. These events are often labeled `inlineSuggest`.
   - If the user explicitly submits the search and sees a results page, the event is typically labeled `resultsPage` in `TRIGGER_MODE`.

2. **The query is captured as entered by the user**
   - This becomes `INPUT_TEXT`.
   - It may contain typos, incomplete words, casing differences, punctuation, or user-specific phrasing.

3. **The query is normalized and possibly corrected upstream**
   - The search stack may lowercase, trim, normalize accents or punctuation, or apply query correction.
   - If the original query would have produced no results, the upstream pipeline may attempt a corrected query.
   - The final string seen by the alternative-search component is stored as `NORMALIZED_TEXT`.

4. **The experiment arm is determined**
   - Users are randomly assigned at the `ACTOR_ID` level to either `Baseline` or `Alternate`.
   - Assignment is stable so the same actor stays in the same arm across events.

5. **The search request is executed**
   - In **Baseline**, the search runs through the production retrieval and ranking pipeline.
   - In **Alternate**, the request may or may not activate the alternative-search component.

6. **Alternate mechanism behavior is recorded**
   - If the Alternate flow runs, it may:
     - use a **warm precomputed store** (`warm_store_hit`)
     - use a **session-level reusable store** (`session_store_hit`)
     - compute a **fresh online representation and save it** (`fresh_compute_saved`)
     - skip auxiliary processing / return no auxiliary response (`no_aux_response`)
   - This becomes `ALT_FLOW_STATUS`.

7. **The request returns a result set**
   - The number of items returned becomes `ITEM_COUNT`.
   - The total response time becomes `LATENCY_MS`.

8. **The user may engage and/or order**
   - If a results page is shown and the user clicks products, the clicked positions are logged in `ENGAGED_POSITIONS`.
   - If purchases are attributed to the request, purchased positions are logged in `ORDERED_POSITIONS` and total request-attributed value is logged in `ORDER_VALUE`. In this context ordered means purchases.

---

## Field-by-field schema

## `COHORT_ID`
- **Type:** string
- **Example values:** `Baseline`, `Alternate`
- **What it is:** experiment arm assignment for the row.
- **How it is generated:** derived from the A/B testing framework based on stable assignment at the actor level.
- **Business context:** this is the main treatment variable used to compare production retrieval with the experimental alternative-search system.

## `ACTOR_ID`
- **Type:** string UUID-like identifier
- **What it is:** user or device-level identifier used as the randomization unit.
- **How it is generated:** created upstream by the analytics or identity system.
- **Business context:** this is the correct clustering unit for statistical inference because treatment assignment is stable at this level.

## `SESSION_KEY`
- **Type:** string
- **What it is:** session identifier grouping multiple request events within a visit.
- **How it is generated:** assigned by the analytics/sessionization layer.
- **Business context:** used to study search journeys, first-request-only views, and repeated reformulations within the same session.

## `REQUEST_KEY`
- **Type:** string UUID-like identifier
- **What it is:** request identifier for the search event.
- **How it is generated:** assigned by the search/event logging system when the request is processed.
- **Business context:** useful for joining to downstream ranking-event or result-inspection systems.

## `EVENT_KEY`
- **Type:** string hash-like identifier
- **What it is:** deduplication key or canonical event identifier.
- **How it is generated:** assigned by the analytics/event layer.
- **Business context:** often used to deduplicate the dataset so one logical request is counted once.

## `REQUEST_TS`
- **Type:** timezone-aware timestamp
- **What it is:** time at which the request started.
- **How it is generated:** captured at request time by the search/event pipeline.
- **Business context:** used for time-series analysis, session ordering, reuse sequencing, and experiment window filtering.

## `LATENCY_MS`
- **Type:** numeric (milliseconds)
- **What it is:** end-to-end latency of the request.
- **How it is generated:** measured by the search infrastructure.
- **Business context:** this is a core latency guardrail. Auxiliary retrieval usually improves coverage or relevance at the cost of higher latency.

## `ITEM_COUNT`
- **Type:** integer
- **What it is:** number of items returned by the request.
- **How it is generated:** emitted by the search backend after retrieval and ranking complete.
- **Business context:** key for no-results and results-presence analyses. A major value proposition of alternative search is reducing zero-result experiences.

## `INPUT_TEXT`
- **Type:** string
- **What it is:** raw search text entered by the user.
- **How it is generated:** captured directly from the frontend request.
- **Business context:** represents user intent as typed, including typos, partial queries, abbreviations, and natural language variation.

## `ENGAGED_POSITIONS`
- **Type:** nullable list of integers
- **Example:** `[1, 3, 2]`
- **What it is:** 1-based positions of products clicked for the request.
- **How it is generated:** recorded from click analytics after the results page is shown and products are clicked.
- **Business context:** used to compute CTR, clicks per request, and ranking-quality proxies such as best engaged position.
- **Important constraint:** if no click occurred, this is `None` or empty.

## `ORDERED_POSITIONS`
- **Type:** nullable list of integers
- **What it is:** 1-based positions of purchased or ordered products attributed to the request. Order in this context is purchases.
- **How it is generated:** joined from order attribution logs.
- **Business context:** lets analysts test whether ranking changes affect downstream commercial outcomes.
- **Important constraint:** this is usually sparse and often a subset of clicked items.

## `ORDER_VALUE`
- **Type:** nullable float
- **What it is:** total purchase value attributed to the request.
- **How it is generated:** order attribution or analytics layer.
- **Business context:** used as a business guardrail. Relevance improvements are less compelling if they hurt monetization.

## `ALT_FLOW_STATUS`
- **Type:** string
- **Typical values:**
  - `no_aux_response`
  - `warm_store_hit`
  - `fresh_compute_saved`
  - `session_store_hit`
- **What it is:** mechanism label describing how the Alternate flow handled the auxiliary retrieval part of the request.
- **How it is generated:** produced by the alternative-search orchestration logic.
- **Business context:** this is critical for mechanism-level analysis.
  - `warm_store_hit`: request representation found in the warm precomputed store
  - `session_store_hit`: request representation previously generated online in a previous session and reused
  - `fresh_compute_saved`: request needed fresh online computation and was saved for later reuse
  - `no_aux_response`: no auxiliary response used because the request was skipped, filtered, failed, or behaved like baseline
- **Important constraint:** in many A/B setups, **Baseline** rows always carry `no_aux_response` because the auxiliary mechanism is not active.

## `NORMALIZED_TEXT`
- **Type:** string
- **What it is:** final normalized or corrected text that reaches the alternative-search component.
- **How it is generated:** produced after query normalization, correction, and filtering logic.
- **Business context:** important for measuring preprocessing and correction burden. Large differences from `INPUT_TEXT` may indicate spell correction or more substantive rewriting.

## `TRIGGER_MODE`
- **Type:** nullable string
- **Typical values:** `inlineSuggest`, `resultsPage`
- **What it is:** frontend interaction context for the request.
- **How it is generated:** supplied by the website or application depending on how the request was triggered.
- **Business context:** this field is crucial because `inlineSuggest` events may call the backend without showing the final results page to the user. Such events often have near-zero CTR by construction and can confound engagement analysis if not handled separately.

---

## Important invariants and relationships

A realistic synthetic dataset should preserve these relationships:

1. **Actor-level treatment assignment**
   - A given `ACTOR_ID` should remain in one arm.

2. **Session structure**
   - Multiple request events may occur within a `SESSION_KEY`.
   - Inline-suggest events may cluster within a session and occur quickly before a visible page request.

3. **Baseline mechanism behavior**
   - Baseline rows should almost always have `ALT_FLOW_STATUS = no_aux_response`.

4. **Trigger-mode effect on engagement**
   - `inlineSuggest` should have near-zero CTR and purchase behavior because users often do not see a clickable page.

5. **Popularity structure**
   - A small set of popular queries should account for a large share of traffic.
   - Tail queries should be numerous and individually sparse.

6. **Mechanism mix differs by popularity**
   - Popular queries should be more likely to hit `warm_store_hit` or `no_aux_response`.
   - Tail queries should be more likely to hit `fresh_compute_saved` and later `session_store_hit`.

7. **Live reuse dependence**
   - `session_store_hit` should only happen for a normalized text that has been seen before through `fresh_compute_saved`.

8. **Latency by mechanism**
   - `warm_store_hit`: moderately elevated latency
   - `session_store_hit`: modestly elevated latency
   - `fresh_compute_saved`: highest latency
   - `no_aux_response`: near-baseline latency

9. **Engagement by mechanism**
   - `warm_store_hit` and `session_store_hit` should typically perform better than `fresh_compute_saved`.
   - `fresh_compute_saved` can show weaker CTR but still improved ranking quality among clicks.

10. **Correction burden**
    - `NORMALIZED_TEXT` may differ from `INPUT_TEXT`, especially for typo-prone or partial queries.

11. **Sparse purchases**
    - Orders should be much rarer than clicks.
    - `ORDER_VALUE` should usually be missing when there is no order.

---

## Business interpretation of the dataset

This dataset is designed to evaluate whether an auxiliary retrieval strategy adds value beyond the standard search stack.

Typical business questions include:
- Does the Alternate flow improve CTR?
- Does it reduce no-result experiences?
- Does it improve ranking quality among clicked items?
- Does it preserve purchase or order-value guardrails?
- Which execution modes create value, and which are risky?
- Are visible improvements concentrated in popular or tail traffic?
- Are observed degradations due to latency, first-touch computation, correction burden, or trigger-mode artifacts?

For testing code, a synthetic generator should therefore preserve not only schema types but also the **causal structure** and **behavioral constraints** analysts expect to find in the real dataset.


In [1]:
import argparse
import ast
import math
import random
import string
import uuid
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

TIMEZONE = "America/Toronto"

BASELINE = "Baseline"
ALTERNATE = "Alternate"

STATUS_NO_AUX = "no_aux_response"
STATUS_WARM = "warm_store_hit"
STATUS_FRESH = "fresh_compute_saved"
STATUS_SESSION = "session_store_hit"

TRIGGER_INLINE = "inlineSuggest"
TRIGGER_RESULTS = "resultsPage"


@dataclass
class GeneratorConfig:
    n_events: int = 100_000
    n_clients: int = 30_000
    start_time: str = "2026-02-12T16:30:03-05:00"
    end_time: str = "2026-03-06T12:16:14-05:00"
    alternate_share: float = 0.50
    head_query_count: int = 5_000
    tail_query_count: int = 60_000
    seed: int = 7
    inline_share_base: float = 0.65
    max_events_per_session: int = 8
    mean_events_per_session: float = 2.8


def norm_text(x: str | None) -> str:
    if x is None:
        return "__NULL__"
    s = str(x).strip().lower()
    return s if s != "" else "__EMPTY__"


def make_uuid() -> str:
    return str(uuid.uuid4())


def make_hashish(n: int = 32) -> str:
    alphabet = string.hexdigits.lower()[:16]
    return "".join(random.choice(alphabet) for _ in range(n))


def poisson_ge_1(lam: float) -> int:
    x = np.random.poisson(lam)
    return int(max(1, x))


def trunc_norm(mean: float, sd: float, lower: float = 0) -> float:
    x = np.random.normal(mean, sd)
    return float(max(lower, x))


def logistic(x: float) -> float:
    return 1.0 / (1.0 + math.exp(-x))


def build_query_catalog(cfg: GeneratorConfig) -> pd.DataFrame:
    rows: List[dict] = []

    head_modifiers = [
        "10x40", "painting", "doorway", "plywood", "screw", "insulation", "window",
        "hinge", "caulk", "dishwasher", "tile", "plank", "drill", "ladder",
        "cabinet", "saw", "bracket", "sealant", "primer", "glue"
    ]
    head_nouns = [
        "wood", "deck", "window", "painting", "shelf", "panel", "vinyl",
        "drywall", "hammer", "plumbing", "heater", "floor", "roof", "wire"
    ]

    for i in range(cfg.head_query_count):
        q = f"{random.choice(head_modifiers)} {random.choice(head_nouns)}"
        rows.append({
            "canonical_text": q,
            "traffic_band": "popular",
            "base_weight": 1.0 / ((i + 1) ** 0.85),
            "base_quality": np.random.uniform(0.2, 0.8),
        })

    tail_tokens_1 = [
        "songue", "bestbuy", "symbol", "champers", "mexican", "trumpet", "moist",
        "symbolist", "congeree", "comedian", "rapid", "composites", "simply",
        "trail", "mouse", "plymouth", "cannon", "semper", "chowder"
    ]
    tail_tokens_2 = [
        "exterior", "cotton", "acer", "nvidia", "black", "carbon",
        "loud", "accounts", "rounded", "large", "trolling", "premium",
        "special", "usage", "train", "decorations", "faraway", "vancouver"
    ]
    tail_tokens_3 = [
        "3.8", "1.2", "4.8", "2.6", "36inches", "48feet", "10mm", "12mm", "6ft",
        "8ft", "24inches", "18meters", "5lb", "20kg", "15lbs"
    ]

    for i in range(cfg.tail_query_count):
        parts = [random.choice(tail_tokens_1), random.choice(tail_tokens_2)]
        if np.random.rand() < 0.6:
            parts.append(random.choice(tail_tokens_3))
        q = " ".join(parts)
        rows.append({
            "canonical_text": q,
            "traffic_band": "non_popular",
            "base_weight": 1.0 / ((i + 1) ** 1.05),
            "base_quality": np.random.uniform(-0.6, 0.4),
        })

    cat = pd.DataFrame(rows).drop_duplicates(subset=["canonical_text"]).reset_index(drop=True)
    cat["bucket_weight"] = cat.groupby("traffic_band")["base_weight"].transform(lambda s: s / s.sum())
    band_multiplier = np.where(cat["traffic_band"] == "popular", 0.85, 0.15)
    cat["weight"] = cat["bucket_weight"] * band_multiplier
    cat["weight"] = cat["weight"] / cat["weight"].sum()
    return cat[["canonical_text", "traffic_band", "weight", "base_quality"]]


def maybe_typo(text: str) -> str:
    if not text:
        return text
    r = np.random.rand()
    if r < 0.35 and len(text) > 3:
        i = np.random.randint(0, len(text) - 1)
        return text[:i] + text[i + 1] + text[i] + text[i + 2:]
    if r < 0.70 and len(text) > 4:
        i = np.random.randint(0, len(text))
        return text[:i] + text[i + 1:]
    if r < 0.90:
        i = np.random.randint(0, len(text))
        c = random.choice(string.ascii_lowercase)
        return text[:i] + c + text[i + 1:]
    words = text.split()
    if words:
        return " ".join(words[:-1]) or text
    return text


def maybe_prefix(text: str) -> str:
    if len(text) <= 3:
        return text
    cut = np.random.randint(max(1, len(text) // 3), len(text))
    return text[:cut].strip() or text


def process_text(input_text: str, canonical_text: str, is_inline: bool) -> Tuple[str, bool]:
    input_norm = norm_text(input_text)
    canonical_norm = norm_text(canonical_text)

    correction_prob = 0.03
    if input_norm != canonical_norm:
        correction_prob = 0.18 if is_inline else 0.10

    if np.random.rand() < correction_prob:
        return canonical_norm, True
    return input_norm, input_norm != canonical_norm


class SyntheticABTestGenerator:
    def __init__(self, cfg: GeneratorConfig):
        self.cfg = cfg
        self.rng = np.random.default_rng(cfg.seed)
        random.seed(cfg.seed)
        np.random.seed(cfg.seed)

        self.start_dt = datetime.fromisoformat(cfg.start_time).astimezone()
        self.end_dt = datetime.fromisoformat(cfg.end_time).astimezone()
        self.catalog = build_query_catalog(cfg)

        self.actor_ids = [make_uuid() for _ in range(cfg.n_clients)]
        alt_mask = self.rng.random(cfg.n_clients) < cfg.alternate_share
        self.actor_cohort: Dict[str, str] = {
            aid: (ALTERNATE if is_alt else BASELINE)
            for aid, is_alt in zip(self.actor_ids, alt_mask)
        }

        self.alternate_session_store_seen: Dict[str, int] = {}
        self.head_query_set = set(self.catalog.loc[self.catalog["traffic_band"] == "popular", "canonical_text"].tolist())

    def sample_actor(self) -> str:
        return random.choice(self.actor_ids)

    def sample_query(self) -> Tuple[str, str, float]:
        idx = self.rng.choice(self.catalog.index.to_numpy(), p=self.catalog["weight"].to_numpy())
        row = self.catalog.loc[idx]
        return row["canonical_text"], row["traffic_band"], float(row["base_quality"])

    def sample_request_time(self) -> datetime:
        total_seconds = int((self.end_dt - self.start_dt).total_seconds())
        offset = int(self.rng.integers(0, max(total_seconds, 1)))
        return self.start_dt + timedelta(seconds=offset, milliseconds=int(self.rng.integers(0, 1000)))

    def sample_trigger_mode(self, event_index_in_session: int, traffic_band: str) -> str:
        p = self.cfg.inline_share_base
        if event_index_in_session == 0:
            p *= 0.7
        else:
            p *= 1.05
        if traffic_band == "non_popular":
            p *= 0.9
        return TRIGGER_INLINE if self.rng.random() < min(max(p, 0.02), 0.95) else TRIGGER_RESULTS

    def assign_mechanism(self, cohort: str, traffic_band: str, normalized_text: str, is_inline: bool) -> str:
        if cohort == BASELINE:
            return STATUS_NO_AUX

        if traffic_band == "popular":
            if is_inline:
                probs = {
                    STATUS_WARM: 0.48,
                    STATUS_NO_AUX: 0.46,
                    STATUS_FRESH: 0.04,
                    STATUS_SESSION: 0.02,
                }
            else:
                probs = {
                    STATUS_WARM: 0.65,
                    STATUS_NO_AUX: 0.28,
                    STATUS_FRESH: 0.04,
                    STATUS_SESSION: 0.03,
                }
        else:
            if normalized_text in self.alternate_session_store_seen and not is_inline:
                probs = {
                    STATUS_SESSION: 0.55,
                    STATUS_FRESH: 0.30,
                    STATUS_WARM: 0.10,
                    STATUS_NO_AUX: 0.05,
                }
            else:
                if is_inline:
                    probs = {
                        STATUS_FRESH: 0.80,
                        STATUS_SESSION: 0.08,
                        STATUS_WARM: 0.06,
                        STATUS_NO_AUX: 0.06,
                    }
                else:
                    probs = {
                        STATUS_FRESH: 0.58,
                        STATUS_SESSION: 0.22,
                        STATUS_WARM: 0.15,
                        STATUS_NO_AUX: 0.05,
                    }

        keys = list(probs.keys())
        vals = np.array(list(probs.values()), dtype=float)
        vals = vals / vals.sum()
        mechanism = keys[self.rng.choice(len(keys), p=vals)]

        if mechanism == STATUS_FRESH:
            self.alternate_session_store_seen[normalized_text] = 1
        return mechanism

    def simulate_item_count(self, cohort: str, mechanism: str, traffic_band: str, is_inline: bool, corrected: bool) -> int:
        base = 40 if traffic_band == "popular" else 10
        if cohort == BASELINE:
            base += 5 if traffic_band == "popular" else 0
        else:
            if mechanism == STATUS_WARM:
                base += 80 if traffic_band == "popular" else 20
            elif mechanism == STATUS_SESSION:
                base += 12 if traffic_band == "popular" else 15
            elif mechanism == STATUS_FRESH:
                base += -10 if traffic_band == "popular" else 3
            elif mechanism == STATUS_NO_AUX:
                base += 0

        if corrected:
            base += 4
        if is_inline:
            base *= 0.95

        mean = max(0.5, base)
        item_count = int(self.rng.negative_binomial(n=3, p=3 / (3 + mean)))
        return max(0, item_count)

    def simulate_latency(self, cohort: str, mechanism: str, is_inline: bool) -> int:
        if cohort == BASELINE:
            mean, sd = 70, 20
        else:
            if mechanism == STATUS_WARM:
                mean, sd = (105, 30)
            elif mechanism == STATUS_SESSION:
                mean, sd = (90, 25)
            elif mechanism == STATUS_FRESH:
                mean, sd = (175, 55)
            else:
                mean, sd = (65, 18)

        if is_inline:
            mean *= 0.9

        return int(round(trunc_norm(mean, sd, lower=15)))

    def engage_probability(self, cohort: str, mechanism: str, traffic_band: str, item_count: int, is_inline: bool, base_quality: float, corrected: bool) -> float:
        if is_inline:
            return 0.004 + (0.002 if mechanism == STATUS_WARM else 0.0)

        x = -0.3 + base_quality
        if traffic_band == "popular":
            x += 1.2
        else:
            x += 0.1

        if item_count == 0:
            x -= 3.5
        elif item_count < 5:
            x -= 0.8
        else:
            x += 0.2

        if cohort == ALTERNATE:
            if mechanism == STATUS_WARM:
                x += 0.12
            elif mechanism == STATUS_SESSION:
                x += 0.18
            elif mechanism == STATUS_FRESH:
                x -= 0.35

        if corrected:
            x += 0.08

        return float(min(max(logistic(x), 0.0001), 0.98))

    def interactions_given_engage(self, mechanism: str, traffic_band: str) -> int:
        lam = 1.6 if traffic_band == "popular" else 1.1
        if mechanism == STATUS_WARM:
            lam += 0.7
        elif mechanism == STATUS_SESSION:
            lam += 0.2
        elif mechanism == STATUS_FRESH:
            lam -= 0.2
        return poisson_ge_1(max(0.3, lam))

    def sample_engaged_positions(self, n_clicks: int, item_count: int, mechanism: str) -> List[int]:
        if item_count <= 0 or n_clicks <= 0:
            return []

        max_rank = min(max(item_count, 1), 200)
        positions = np.arange(1, max_rank + 1)

        if mechanism == STATUS_WARM:
            alpha = 1.30
        elif mechanism == STATUS_SESSION:
            alpha = 1.45
        elif mechanism == STATUS_FRESH:
            alpha = 1.60
        else:
            alpha = 1.05

        probs = 1 / (positions ** alpha)
        probs = probs / probs.sum()

        size = min(n_clicks, len(positions))
        ranks = self.rng.choice(positions, size=size, replace=False, p=probs)
        return list(map(int, ranks.tolist()))

    def order_probability(self, engaged_any: bool, traffic_band: str, mechanism: str, is_inline: bool) -> float:
        if is_inline or not engaged_any:
            return 0.00002
        p = 0.0008 if traffic_band == "popular" else 0.0005
        if mechanism == STATUS_WARM:
            p += 0.00015
        elif mechanism == STATUS_SESSION:
            p += 0.00010
        elif mechanism == STATUS_FRESH:
            p -= 0.00010
        return max(0.0, p)

    def order_value(self) -> float:
        return float(np.random.lognormal(mean=4.5, sigma=0.45))

    def generate(self) -> pd.DataFrame:
        rows: List[dict] = []

        while len(rows) < self.cfg.n_events:
            actor_id = self.sample_actor()
            cohort = self.actor_cohort[actor_id]
            session_key = make_hashish(32)
            session_start = self.sample_request_time()
            n_events_in_session = min(self.cfg.max_events_per_session, poisson_ge_1(self.cfg.mean_events_per_session))

            last_canonical: Optional[str] = None
            for j in range(n_events_in_session):
                if len(rows) >= self.cfg.n_events:
                    break

                canonical_text, traffic_band, base_quality = self.sample_query()
                if last_canonical is not None and np.random.rand() < 0.25:
                    canonical_text = last_canonical
                    traffic_band = "popular" if canonical_text in self.head_query_set else "non_popular"
                    base_quality = float(self.catalog.loc[self.catalog["canonical_text"] == canonical_text, "base_quality"].iloc[0])

                trigger_mode = self.sample_trigger_mode(j, traffic_band)
                is_inline = trigger_mode == TRIGGER_INLINE

                if is_inline:
                    if np.random.rand() < 0.45:
                        input_text = maybe_prefix(canonical_text)
                    elif np.random.rand() < 0.75:
                        input_text = maybe_typo(canonical_text)
                    else:
                        input_text = canonical_text
                else:
                    input_text = maybe_typo(canonical_text) if np.random.rand() < 0.08 else canonical_text

                normalized_text, corrected = process_text(input_text, canonical_text, is_inline)

                mechanism = self.assign_mechanism(cohort, traffic_band, normalized_text, is_inline)
                request_ts = session_start + timedelta(seconds=float(abs(np.random.normal(loc=1.0 * j, scale=0.8))))
                item_count = self.simulate_item_count(cohort, mechanism, traffic_band, is_inline, corrected)
                latency_ms = self.simulate_latency(cohort, mechanism, is_inline)

                p_engage = self.engage_probability(cohort, mechanism, traffic_band, item_count, is_inline, base_quality, corrected)
                engaged_any = self.rng.random() < p_engage
                n_clicks = self.interactions_given_engage(mechanism, traffic_band) if engaged_any else 0
                engaged_positions = self.sample_engaged_positions(n_clicks, item_count, mechanism)

                p_order = self.order_probability(engaged_any, traffic_band, mechanism, is_inline)
                ordered_any = self.rng.random() < p_order
                if ordered_any and engaged_positions:
                    k = min(len(engaged_positions), poisson_ge_1(1.0))
                    ordered_positions = random.sample(engaged_positions, k=k)
                    order_value = self.order_value()
                else:
                    ordered_positions = None
                    order_value = np.nan

                rows.append({
                    "COHORT_ID": cohort,
                    "ACTOR_ID": actor_id,
                    "SESSION_KEY": session_key,
                    "REQUEST_KEY": make_uuid(),
                    "EVENT_KEY": make_hashish(32),
                    "REQUEST_TS": pd.Timestamp(request_ts).tz_localize(None).tz_localize(request_ts.tzinfo),
                    "LATENCY_MS": latency_ms,
                    "ITEM_COUNT": item_count,
                    "INPUT_TEXT": input_text,
                    "ENGAGED_POSITIONS": engaged_positions if engaged_positions else None,
                    "ORDERED_POSITIONS": ordered_positions,
                    "ORDER_VALUE": order_value,
                    "ALT_FLOW_STATUS": mechanism if cohort == ALTERNATE else STATUS_NO_AUX,
                    "NORMALIZED_TEXT": normalized_text,
                    "TRIGGER_MODE": trigger_mode,
                })

                last_canonical = canonical_text

        df = pd.DataFrame(rows)
        df = df.sort_values(["REQUEST_TS", "ACTOR_ID", "SESSION_KEY"], kind="mergesort").reset_index(drop=True)
        return df


def summarize(df: pd.DataFrame) -> pd.DataFrame:
    out = []
    for cohort, g in df.groupby("COHORT_ID", dropna=False):
        engaged_any = g["ENGAGED_POSITIONS"].apply(lambda x: bool(x) if isinstance(x, list) else False)
        out.append({
            "COHORT_ID": cohort,
            "n_events": len(g),
            "n_actors": g["ACTOR_ID"].nunique(),
            "ctr": engaged_any.mean(),
            "avg_item_count": pd.to_numeric(g["ITEM_COUNT"], errors="coerce").mean(),
            "latency_mean_ms": pd.to_numeric(g["LATENCY_MS"], errors="coerce").mean(),
            "inline_share": g["TRIGGER_MODE"].eq(TRIGGER_INLINE).mean(),
        })
    return pd.DataFrame(out)



print("Generate synthetic AB test search-event data.")


N_EVENTS = 1000
N_CLIENTS = 200
SEED = 7

cfg = GeneratorConfig(
    n_events=N_EVENTS,
    n_clients=N_CLIENTS,
    seed=SEED,
)

gen = SyntheticABTestGenerator(cfg)
df = gen.generate()

print(summarize(df).to_string(index=False))


df.head()

Generate synthetic AB test search-event data.
COHORT_ID  n_events  n_actors      ctr  avg_item_count  latency_mean_ms  inline_share
Alternate       494        78 0.321862       69.676113        95.085020      0.574899
 Baseline       506        91 0.276680       41.227273        64.252964      0.636364


,COHORT_ID,ACTOR_ID,SESSION_KEY,REQUEST_KEY,EVENT_KEY,REQUEST_TS,LATENCY_MS,ITEM_COUNT,INPUT_TEXT,ENGAGED_POSITIONS,ORDERED_POSITIONS,ORDER_VALUE,ALT_FLOW_STATUS,NORMALIZED_TEXT,TRIGGER_MODE
0,Baseline,57bdc362-2ad1-49df-833d-6ba10848160f,ba0d80b5dfb2d1abad2e048a7a240d73,f74de95b-5055-4175-b564-a0dd35ed7a00,8b7fae7f50b534349124154a8b3778b7,2026-02-12 17:08:17.151676-05:00,80,19,tile windwo,None,None,NaN,no_aux_response,tile windwo,inlineSuggest
1,Alternate,35740aa6-3af1-4be0-b5dc-881c26c067d7,b0f597fc483e1be364eeaa94745c29c7,6163f4dd-d8aa-4041-9962-b3c75a9cf562,b1a71ad871cccc1849bf13f1150157f3,2026-02-12 17:36:48.063696-05:00,96,207,plani plumbing,None,None,NaN,warm_store_hit,plani plumbing,inlineSuggest
2,Alternate,35740aa6-3af1-4be0-b5dc-881c26c067d7,b0f597fc483e1be364eeaa94745c29c7,32249474-c1f3-4f06-ba20-8941d5f6e787,bf82996d8d84ac451ee8483136d70e02,2026-02-12 17:36:48.210713-05:00,169,13,cannon carbon 4.8,None,None,NaN,fresh_compute_saved,cannon carbon 4.8,resultsPage
3,Alternate,35740aa6-3af1-4be0-b5dc-881c26c067d7,b0f597fc483e1be364eeaa94745c29c7,befbb772-f994-4577-9959-146e42f745ea,e43980bbb86e36ec382d77fdb7678eb6,2026-02-12 17:36:49.912113-05:00,152,11,cannon carb,None,None,NaN,fresh_compute_saved,cannon carb,inlineSuggest
4,Baseline,8f40c61f-4897-4a10-8b36-a9c2371948cf,e00a689069d3aef64f6d1813aaf893e1,c7c71cf2-14e7-4cca-818b-53e3c5ba2a28,5566f9e8a620d3ae16ea1a0e9478bf50,2026-02-12 19:38:44.397555-05:00,40,10,painting,None,None,NaN,no_aux_response,painting,inlineSuggest


In [2]:
df

,COHORT_ID,ACTOR_ID,SESSION_KEY,REQUEST_KEY,EVENT_KEY,REQUEST_TS,LATENCY_MS,ITEM_COUNT,INPUT_TEXT,ENGAGED_POSITIONS,ORDERED_POSITIONS,ORDER_VALUE,ALT_FLOW_STATUS,NORMALIZED_TEXT,TRIGGER_MODE
0,Baseline,57bdc362-2ad1-49df-833d-6ba10848160f,ba0d80b5dfb2d1abad2e048a7a240d73,f74de95b-5055-4175-b564-a0dd35ed7a00,8b7fae7f50b534349124154a8b3778b7,2026-02-12 17:08:17.151676-05:00,80,19,tile windwo,None,None,NaN,no_aux_response,tile windwo,inlineSuggest
1,Alternate,35740aa6-3af1-4be0-b5dc-881c26c067d7,b0f597fc483e1be364eeaa94745c29c7,6163f4dd-d8aa-4041-9962-b3c75a9cf562,b1a71ad871cccc1849bf13f1150157f3,2026-02-12 17:36:48.063696-05:00,96,207,plani plumbing,None,None,NaN,warm_store_hit,plani plumbing,inlineSuggest
2,Alternate,35740aa6-3af1-4be0-b5dc-881c26c067d7,b0f597fc483e1be364eeaa94745c29c7,32249474-c1f3-4f06-ba20-8941d5f6e787,bf82996d8d84ac451ee8483136d70e02,2026-02-12 17:36:48.210713-05:00,169,13,cannon carbon 4.8,None,None,NaN,fresh_compute_saved,cannon carbon 4.8,resultsPage
3,Alternate,35740aa6-3af1-4be0-b5dc-881c26c067d7,b0f597fc483e1be364eeaa94745c29c7,befbb772-f994-4577-9959-146e42f745ea,e43980bbb86e36ec382d77fdb7678eb6,2026-02-12 17:36:49.912113-05:00,152,11,cannon carb,None,None,NaN,fresh_compute_saved,cannon carb,inlineSuggest
4,Baseline,8f40c61f-4897-4a10-8b36-a9c2371948cf,e00a689069d3aef64f6d1813aaf893e1,c7c71cf2-14e7-4cca-818b-53e3c5ba2a28,5566f9e8a620d3ae16ea1a0e9478bf50,2026-02-12 19:38:44.397555-05:00,40,10,painting,None,None,NaN,no_aux_response,painting,inlineSuggest
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,Baseline,61a953ba-07ea-4aee-a1e8-349cec6360f6,66a305301bba438cf6c1cf356347e955,c24649c1-0d5c-477d-b09a-88f1fac8b4be,71e7121cc80a666b8a9c1e55e234da3a,2026-03-06 09:55:33.222498-05:00,65,76,window wire,"[1, 7]",None,NaN,no_aux_response,window wire,resultsPage
996,Baseline,94b93944-4a66-48fc-a031-3425056faada,6c7ca08280a69ada19c80615155d05ff,7cc99f72-af7a-4840-bf94-dc2fba07834f,dff0060405d5e85e315ca36128e45c43,2026-03-06 10:14:03.886161-05:00,84,33,10x40 wo,None,None,NaN,no_aux_response,10x40 wo,inlineSuggest
997,Baseline,94b93944-4a66-48fc-a031-3425056faada,6c7ca08280a69ada19c80615155d05ff,75ae9034-2664-4d95-ac61-cefefacb583d,a7a1c2438501ad5e0e89aae4b1873aff,2026-03-06 10:14:04.451053-05:00,67,24,10x40 wood,None,None,NaN,no_aux_response,10x40 wood,resultsPage
998,Baseline,e49d24da-c72f-4a30-8dc3-ef2980490791,d4575625da165b918443a09fda07f1ba,6fe812fe-3580-4210-8919-3049eb0fffc7,f2d142708d750744960dce784c00ab8d,2026-03-06 11:47:37.155835-05:00,61,34,plywoo dpanel,None,None,NaN,no_aux_response,plywoo dpanel,inlineSuggest


In [4]:
import os

# Use '../' to move out of the notebooks folder into the repo root
output_dir = '../docs/simulated_data/'
output_file = os.path.join(output_dir, 'scenario_1.csv')

# Create the directory at the root level if it doesn't exist
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"Created directory at: {os.path.abspath(output_dir)}")

# Export the dataframe
df.to_csv(output_file, index=False)

print(f"Success! Dataset saved to: {output_file}")

Created directory at: /Users/nicolas.cabrera-malik/Documents/claude-internal-skills/docs/simulated_data
Success! Dataset saved to: ../docs/simulated_data/scenario_1.csv


# Synthetic A/B Test Mechanism Playbook

## Purpose

This document explains how to inject realistic mechanisms into the current synthetic A/B test generator so downstream A/B analysis code can be validated against known ground truth.

The goal is not to reproduce one exact production experiment, but to create controlled synthetic scenarios that mimic the kinds of effects seen in experiments involving a baseline retrieval flow and an **alternative search** flow.

This revision is intentionally explicit. For each scenario, it tells you:
- **which function in the current notebook code to edit**
- **which block inside that function to replace or extend**
- **what kind of visible effect the change should create in the generated data**

---

## Current generator structure

The current notebook code already simulates several useful effects:
- stable actor-level randomization into `Baseline` and `Alternate`
- head vs tail query traffic
- `inlineSuggest` vs `resultsPage` trigger modes
- four alternate-flow statuses:
  - `warm_store_hit`
  - `session_store_hit`
  - `fresh_compute_saved`
  - `no_aux_response`
- different latency, item-count, engagement, and order behavior by mechanism

The most important functions in the current code are:
- `sample_trigger_mode(...)`
- `assign_mechanism(...)`
- `simulate_item_count(...)`
- `simulate_latency(...)`
- `engage_probability(...)`
- `interactions_given_engage(...)`
- `sample_engaged_positions(...)`
- `order_probability(...)`
- `generate(...)`

---

## How to read the scenarios below

Each scenario is written against the **current notebook implementation**, not against a hypothetical future refactor.

When a scenario says something like:
- “modify `assign_mechanism(...)`”
- “replace the `if is_inline:` block inside `engage_probability(...)`”

that means:
1. find the function in your current notebook code
2. find the exact block mentioned
3. replace or adjust that block in place
4. rerun `gen = SyntheticABTestGenerator(cfg)` and `df = gen.generate()`

Unless explicitly stated, you do **not** need to add new tables, helper classes, or external modules.

---

## Optional but recommended: add scenario knobs to `GeneratorConfig`

You can hard-code scenario changes directly in the functions, but a cleaner pattern is to add knobs to `GeneratorConfig` first.

Add fields like these to `GeneratorConfig`:

```python
@dataclass
class GeneratorConfig:
    n_events: int = 100_000
    n_clients: int = 30_000
    start_time: str = "2026-02-12T16:30:03-05:00"
    end_time: str = "2026-03-06T12:16:14-05:00"
    alternate_share: float = 0.50
    head_query_count: int = 5_000
    tail_query_count: int = 60_000
    seed: int = 7
    inline_share_base: float = 0.65
    max_events_per_session: int = 8
    mean_events_per_session: float = 2.8

    # scenario knobs
    inline_ctr_level: float = 0.004
    inline_warm_ctr_bonus: float = 0.002

    warm_ctr_bonus: float = 0.12
    session_ctr_bonus: float = 0.18
    fresh_ctr_penalty: float = -0.35

    warm_item_bonus_head: int = 80
    warm_item_bonus_tail: int = 20
    session_item_bonus_tail: int = 15
    fresh_item_penalty_head: int = -10

    warm_latency_mean: float = 105
    session_latency_mean: float = 90
    fresh_latency_mean: float = 175

    corrected_query_bonus: float = 0.08
```

Then reference `self.cfg.<field>` inside the functions below.

---

# Mechanism catalog

## 1) Strong warm-store success

### What it simulates
A scenario where `warm_store_hit` is the healthiest mode of the alternative search system.

### Why it matters
This validates:
- static-cache matched views
- head/popular query improvements
- positive CTR and clicks/search without severe ranking harm

### Exact functions to edit
- `simulate_item_count(...)`
- `engage_probability(...)`
- optionally `simulate_latency(...)`

### Exact place to change in `simulate_item_count(...)`
Find this block:

```python
if cohort == BASELINE:
    base += 5 if traffic_band == "popular" else 0
else:
    if mechanism == STATUS_WARM:
        base += 80 if traffic_band == "popular" else 20
    elif mechanism == STATUS_SESSION:
        base += 12 if traffic_band == "popular" else 15
    elif mechanism == STATUS_FRESH:
        base += -10 if traffic_band == "popular" else 3
    elif mechanism == STATUS_NO_AUX:
        base += 0
```

Replace the warm-store line with a stronger effect, for example:

```python
if cohort == BASELINE:
    base += 5 if traffic_band == "popular" else 0
else:
    if mechanism == STATUS_WARM:
        base += 110 if traffic_band == "popular" else 35
    elif mechanism == STATUS_SESSION:
        base += 12 if traffic_band == "popular" else 15
    elif mechanism == STATUS_FRESH:
        base += -10 if traffic_band == "popular" else 3
    elif mechanism == STATUS_NO_AUX:
        base += 0
```

### Exact place to change in `engage_probability(...)`
Find this block:

```python
if cohort == ALTERNATE:
    if mechanism == STATUS_WARM:
        x += 0.12
    elif mechanism == STATUS_SESSION:
        x += 0.18
    elif mechanism == STATUS_FRESH:
        x -= 0.35
```

Make `STATUS_WARM` stronger, for example:

```python
if cohort == ALTERNATE:
    if mechanism == STATUS_WARM:
        x += 0.22
    elif mechanism == STATUS_SESSION:
        x += 0.18
    elif mechanism == STATUS_FRESH:
        x -= 0.35
```

### Optional place to change in `simulate_latency(...)`
Find:

```python
if mechanism == STATUS_WARM:
    mean, sd = (105, 30)
```

Reduce latency slightly if you want warm-store to look both better and faster:

```python
if mechanism == STATUS_WARM:
    mean, sd = (95, 24)
```

### Expected visible effect
- `warm_store_hit` matched view shows positive CTR and clicks/search
- results presence improves
- no-results declines
- latency remains moderate

---

## 2) Strong session-store reuse effect

### What it simulates
A scenario where `session_store_hit` becomes the strongest tail-query mechanism once a normalized query has been seen before.

### Why it matters
This validates:
- tail-query improvements
- live-cache matched analysis
- the idea that reuse is stronger than first-touch computation

### Exact functions to edit
- `assign_mechanism(...)`
- `engage_probability(...)`
- `sample_engaged_positions(...)`
- optionally `simulate_item_count(...)`

### Exact place to change in `assign_mechanism(...)`
Find this block inside the non-popular branch:

```python
if normalized_text in self.alternate_session_store_seen and not is_inline:
    probs = {
        STATUS_SESSION: 0.55,
        STATUS_FRESH: 0.30,
        STATUS_WARM: 0.10,
        STATUS_NO_AUX: 0.05,
    }
```

Make session-store more dominant:

```python
if normalized_text in self.alternate_session_store_seen and not is_inline:
    probs = {
        STATUS_SESSION: 0.72,
        STATUS_FRESH: 0.15,
        STATUS_WARM: 0.10,
        STATUS_NO_AUX: 0.03,
    }
```

### Exact place to change in `engage_probability(...)`
Find:

```python
elif mechanism == STATUS_SESSION:
    x += 0.18
```

Increase it, for example:

```python
elif mechanism == STATUS_SESSION:
    x += 0.28
```

### Exact place to change in `sample_engaged_positions(...)`
Find:

```python
elif mechanism == STATUS_SESSION:
    alpha = 1.45
```

Increase rank concentration if you want better top-rank clicking:

```python
elif mechanism == STATUS_SESSION:
    alpha = 1.65
```

### Optional place to change in `simulate_item_count(...)`
Find:

```python
elif mechanism == STATUS_SESSION:
    base += 12 if traffic_band == "popular" else 15
```

Strengthen the tail effect:

```python
elif mechanism == STATUS_SESSION:
    base += 12 if traffic_band == "popular" else 25
```

### Expected visible effect
- `session_store_hit` matched view shows strong CTR uplift
- clicked ranks improve
- non-popular queries improve once they become reusable

---

## 3) Weak first-touch fresh-compute path

### What it simulates
A scenario where `fresh_compute_saved` is slower and weaker than both `warm_store_hit` and `session_store_hit`.

### Why it matters
This validates:
- the idea that first-touch computation is the weakest mechanism
- latency vs engagement tradeoff analysis
- clean-slice live-embed matched views

### Exact functions to edit
- `simulate_latency(...)`
- `simulate_item_count(...)`
- `engage_probability(...)`
- optionally `order_probability(...)`

### Exact place to change in `simulate_latency(...)`
Find:

```python
elif mechanism == STATUS_FRESH:
    mean, sd = (175, 55)
```

Make it worse:

```python
elif mechanism == STATUS_FRESH:
    mean, sd = (215, 65)
```

### Exact place to change in `simulate_item_count(...)`
Find:

```python
elif mechanism == STATUS_FRESH:
    base += -10 if traffic_band == "popular" else 3
```

Make fresh-compute worse on both head and tail:

```python
elif mechanism == STATUS_FRESH:
    base += -18 if traffic_band == "popular" else -2
```

### Exact place to change in `engage_probability(...)`
Find:

```python
elif mechanism == STATUS_FRESH:
    x -= 0.35
```

Make it substantially weaker:

```python
elif mechanism == STATUS_FRESH:
    x -= 0.60
```

### Optional place to change in `order_probability(...)`
Find:

```python
elif mechanism == STATUS_FRESH:
    p -= 0.00010
```

Make fresh-compute weaker downstream too:

```python
elif mechanism == STATUS_FRESH:
    p -= 0.00020
```

### Expected visible effect
- `fresh_compute_saved` matched view shows lower CTR and clicks/search
- results presence may stagnate or decline
- latency jumps sharply
- best-click rank can still improve if click concentration remains top-heavy

---

## 4) Searchbox contamination / inline-trigger bias

### What it simulates
A scenario where many alternate-flow events happen under `inlineSuggest`, meaning the retrieval layer is called but the page is not actually shown in a user-clickable way.

### Why it matters
This validates:
- trigger-mode filtering logic
- action-cause sanity checks
- the possibility that a mechanism looks weak only because it is overrepresented in `inlineSuggest`

### Exact functions to edit
- `sample_trigger_mode(...)`
- `assign_mechanism(...)`
- `engage_probability(...)`

### Exact place to change in `sample_trigger_mode(...)`
Find:

```python
def sample_trigger_mode(self, event_index_in_session: int, traffic_band: str) -> str:
    p = self.cfg.inline_share_base
    if event_index_in_session == 0:
        p *= 0.7
    else:
        p *= 1.05
    if traffic_band == "non_popular":
        p *= 0.9
    return TRIGGER_INLINE if self.rng.random() < min(max(p, 0.02), 0.95) else TRIGGER_RESULTS
```

If you want more inline traffic overall, raise `p`, for example:

```python
def sample_trigger_mode(self, event_index_in_session: int, traffic_band: str) -> str:
    p = self.cfg.inline_share_base
    if event_index_in_session == 0:
        p *= 0.9
    else:
        p *= 1.20
    if traffic_band == "non_popular":
        p *= 1.05
    return TRIGGER_INLINE if self.rng.random() < min(max(p, 0.02), 0.98) else TRIGGER_RESULTS
```

### Exact place to change in `assign_mechanism(...)`
Find the non-popular + inline branch:

```python
else:
    if is_inline:
        probs = {
            STATUS_FRESH: 0.80,
            STATUS_SESSION: 0.08,
            STATUS_WARM: 0.06,
            STATUS_NO_AUX: 0.06,
        }
```

This is already quite inline-heavy for fresh compute. If you want even stronger contamination, increase it further:

```python
else:
    if is_inline:
        probs = {
            STATUS_FRESH: 0.90,
            STATUS_SESSION: 0.04,
            STATUS_WARM: 0.03,
            STATUS_NO_AUX: 0.03,
        }
```

### Exact place to change in `engage_probability(...)`
Find the first early-return block:

```python
if is_inline:
    return 0.004 + (0.002 if mechanism == STATUS_WARM else 0.0)
```

Make inline engagement near zero:

```python
if is_inline:
    return 0.002 + (0.001 if mechanism == STATUS_WARM else 0.0)
```

### Expected visible effect
- raw mechanism CTRs for `fresh_compute_saved` look extremely poor
- clean-slice reruns materially change conclusions
- `inlineSuggest` rows have near-zero engagement across mechanisms

---

## 5) Query-correction burden concentrated in fresh-compute

### What it simulates
A scenario where `fresh_compute_saved` receives more typo-heavy or partially typed input, causing `INPUT_TEXT` to differ more often from `NORMALIZED_TEXT`.

### Why it matters
This validates:
- query-correction burden analyses
- hypotheses about whether a weak mechanism simply gets harder inputs

### Exact functions to edit
- `generate(...)`
- `process_text(...)`

### Exact place to change in `generate(...)`
Find this block:

```python
if is_inline:
    if np.random.rand() < 0.45:
        input_text = maybe_prefix(canonical_text)
    elif np.random.rand() < 0.75:
        input_text = maybe_typo(canonical_text)
    else:
        input_text = canonical_text
else:
    input_text = maybe_typo(canonical_text) if np.random.rand() < 0.08 else canonical_text
```

To create more difficult inputs, especially for tail traffic, make this more aggressive:

```python
if is_inline:
    if np.random.rand() < 0.60:
        input_text = maybe_prefix(canonical_text)
    elif np.random.rand() < 0.92:
        input_text = maybe_typo(canonical_text)
    else:
        input_text = canonical_text
else:
    if traffic_band == "non_popular":
        input_text = maybe_typo(canonical_text) if np.random.rand() < 0.18 else canonical_text
    else:
        input_text = maybe_typo(canonical_text) if np.random.rand() < 0.08 else canonical_text
```

### Exact place to change in `process_text(...)`
Find:

```python
correction_prob = 0.03
if input_norm != canonical_norm:
    correction_prob = 0.18 if is_inline else 0.10
```

Make correction more common:

```python
correction_prob = 0.05
if input_norm != canonical_norm:
    correction_prob = 0.28 if is_inline else 0.16
```

### Expected visible effect
- `INPUT_TEXT` and `NORMALIZED_TEXT` differ more often
- correction-burden diagnostics show heavier rewrite load
- if paired with a fresh-compute penalty, the mechanism may look weak partly due to harder inputs

---

## 6) Coverage win with weaker ranking precision

### What it simulates
A scenario where the alternative search increases `ITEM_COUNT` and lowers no-results, but the click quality degrades because relevant items are found deeper in the ranking.

### Why it matters
This validates:
- result-presence wins with weaker clicked-rank proxies
- non-popular coverage improvements that do not fully convert into quality gains

### Exact functions to edit
- `simulate_item_count(...)`
- `sample_engaged_positions(...)`
- optionally `engage_probability(...)`

### Exact place to change in `simulate_item_count(...)`
Increase item counts for a mechanism, for example fresh-compute or session-store:

```python
elif mechanism == STATUS_FRESH:
    base += 10 if traffic_band == "popular" else 18
```

### Exact place to change in `sample_engaged_positions(...)`
Find:

```python
elif mechanism == STATUS_FRESH:
    alpha = 1.60
```

Reduce rank concentration so clicks come from deeper positions:

```python
elif mechanism == STATUS_FRESH:
    alpha = 1.10
```

Lower `alpha` means flatter position probabilities and worse clicked-rank behavior.

### Expected visible effect
- results presence rises
- no-results falls
- CTR may improve modestly or stay flat
- best-click-top10 and best clicked rank worsen

---

## 7) Late-period divergence or scoring drift

### What it simulates
A scenario where the experiment behaves normally early on, but the alternate flow changes during the last few days.

### Why it matters
This validates:
- daily metric monitoring
- detection of late CTR divergence
- time-window sensitivity checks

### Exact functions to edit
- `__init__(...)`
- `engage_probability(...)`
- optionally `simulate_item_count(...)`

### Exact place to change in `__init__(...)`
Add a late-period cutoff after the existing datetime setup:

```python
self.late_period_start = datetime.fromisoformat("2026-03-02T00:00:00-05:00").astimezone()
```

### Exact place to change in `engage_probability(...)`
To use time in the function, first change the function signature.

Find:

```python
def engage_probability(self, cohort: str, mechanism: str, traffic_band: str, item_count: int, is_inline: bool, base_quality: float, corrected: bool) -> float:
```

Replace with:

```python
def engage_probability(self, cohort: str, mechanism: str, traffic_band: str, item_count: int, is_inline: bool, base_quality: float, corrected: bool, request_ts: datetime) -> float:
```

Then, near the end of the function, add a late-period shift:

```python
if cohort == ALTERNATE and request_ts >= self.late_period_start and traffic_band == "popular":
    x -= 0.25
```

### Exact place to change in `generate(...)`
Because the signature changed, find this line:

```python
p_engage = self.engage_probability(cohort, mechanism, traffic_band, item_count, is_inline, base_quality, corrected)
```

Replace with:

```python
p_engage = self.engage_probability(cohort, mechanism, traffic_band, item_count, is_inline, base_quality, corrected, request_ts)
```

### Expected visible effect
- daily CTR plots diverge late in the experiment
- the shift can be concentrated in head/popular traffic if desired

---

## 8) Under-activated head traffic

### What it simulates
A scenario where head traffic is heavily routed to `no_aux_response`, making the alternate system look muted on popular queries despite good performance in active cached modes.

### Why it matters
This validates:
- exposure dilution in popular traffic
- muted head-query uplift despite strong active mechanisms

### Exact functions to edit
- `assign_mechanism(...)`

### Exact place to change in `assign_mechanism(...)`
Find the popular branches.

Current inline popular branch:

```python
if traffic_band == "popular":
    if is_inline:
        probs = {
            STATUS_WARM: 0.48,
            STATUS_NO_AUX: 0.46,
            STATUS_FRESH: 0.04,
            STATUS_SESSION: 0.02,
        }
```

Make it even more under-activated:

```python
if traffic_band == "popular":
    if is_inline:
        probs = {
            STATUS_WARM: 0.35,
            STATUS_NO_AUX: 0.60,
            STATUS_FRESH: 0.03,
            STATUS_SESSION: 0.02,
        }
```

Current visible popular branch:

```python
else:
    probs = {
        STATUS_WARM: 0.65,
        STATUS_NO_AUX: 0.28,
        STATUS_FRESH: 0.04,
        STATUS_SESSION: 0.03,
    }
```

Make visible popular traffic under-activated too:

```python
else:
    probs = {
        STATUS_WARM: 0.48,
        STATUS_NO_AUX: 0.45,
        STATUS_FRESH: 0.04,
        STATUS_SESSION: 0.03,
    }
```

### Expected visible effect
- overall popular-query uplift looks weak
- `warm_store_hit` still looks strong when isolated
- mechanism-share tables show large `no_aux_response` volume on popular queries

---

## 9) Search-journey amplification

### What it simulates
A scenario where the main gains emerge through later events in the same session rather than the first visible search.

### Why it matters
This validates:
- first-search-only sensitivity analyses
- search-per-visit and session-depth patterns

### Exact functions to edit
- `sample_trigger_mode(...)`
- `generate(...)`
- optionally `engage_probability(...)`

### Exact place to change in `sample_trigger_mode(...)`
Make later events more likely to be visible page searches:

Find:

```python
if event_index_in_session == 0:
    p *= 0.7
else:
    p *= 1.05
```

Replace with:

```python
if event_index_in_session == 0:
    p *= 0.9
else:
    p *= 0.75
```

This reduces inline share later in the session.

### Exact place to change in `generate(...)`
Use `last_canonical` more often to create reformulation chains. Find:

```python
if last_canonical is not None and np.random.rand() < 0.25:
    canonical_text = last_canonical
```

Increase this probability:

```python
if last_canonical is not None and np.random.rand() < 0.45:
    canonical_text = last_canonical
```

### Optional place to change in `engage_probability(...)`
If you want later events to be more successful, add `event_index_in_session` as an argument and boost non-first events. That requires:
- changing the function signature
- passing `j` from `generate(...)`

### Expected visible effect
- all-search analysis looks stronger than first-search-only analysis
- repeated sessions / reformulations become more common

---

# Practical implementation notes

## Where to make changes safely
The safest workflow is:
1. copy the current notebook cell to a new version
2. change only one mechanism at a time
3. rerun:
   ```python
   gen = SyntheticABTestGenerator(cfg)
   df = gen.generate()
   summarize(df)
   ```
4. inspect whether the expected effect appeared

## Which function controls what
- **traffic routing**: `sample_trigger_mode(...)`
- **mechanism assignment**: `assign_mechanism(...)`
- **coverage / item count**: `simulate_item_count(...)`
- **latency**: `simulate_latency(...)`
- **CTR-like behavior**: `engage_probability(...)`
- **click multiplicity**: `interactions_given_engage(...)`
- **clicked-rank behavior**: `sample_engaged_positions(...)`
- **order conversion**: `order_probability(...)`
- **session structure / query sequences**: `generate(...)`

---

# Helper functions: approximate sample size calculations

These helper functions estimate approximate required sample size for common metric types. They are intended for simulation planning, not for replacing formal power analysis.

## 1) Binary metric sample size (CTR, order rate, results-present rate)

```python
import math
from statistics import NormalDist


def approx_n_per_arm_binary(
    baseline_rate: float,
    target_rate: float,
    alpha: float = 0.05,
    power: float = 0.80,
) -> int:
    """
    Approximate per-arm sample size for a two-sided test of two proportions.
    Uses a standard normal approximation.
    """
    p1 = baseline_rate
    p2 = target_rate
    if not (0 < p1 < 1 and 0 < p2 < 1):
        raise ValueError("Rates must be between 0 and 1.")

    z_alpha = NormalDist().inv_cdf(1 - alpha / 2)
    z_beta = NormalDist().inv_cdf(power)

    p_bar = 0.5 * (p1 + p2)
    delta = abs(p2 - p1)
    if delta == 0:
        raise ValueError("Target effect must be non-zero.")

    term1 = z_alpha * math.sqrt(2 * p_bar * (1 - p_bar))
    term2 = z_beta * math.sqrt(p1 * (1 - p1) + p2 * (1 - p2))

    n = ((term1 + term2) ** 2) / (delta ** 2)
    return math.ceil(n)
```

### Example

```python
# Baseline CTR 0.12, target CTR 0.126 (+0.6 pp)
approx_n_per_arm_binary(0.12, 0.126, alpha=0.05, power=0.80)
```

---

## 2) Continuous metric sample size (latency mean, average item count)

```python
import math
from statistics import NormalDist


def approx_n_per_arm_continuous(
    baseline_mean: float,
    target_mean: float,
    sd: float,
    alpha: float = 0.05,
    power: float = 0.80,
) -> int:
    """
    Approximate per-arm sample size for comparing two means.
    Assumes equal variance and a normal approximation.
    """
    delta = abs(target_mean - baseline_mean)
    if delta == 0:
        raise ValueError("Target effect must be non-zero.")
    if sd <= 0:
        raise ValueError("sd must be positive.")

    z_alpha = NormalDist().inv_cdf(1 - alpha / 2)
    z_beta = NormalDist().inv_cdf(power)

    n = 2 * ((z_alpha + z_beta) ** 2) * (sd ** 2) / (delta ** 2)
    return math.ceil(n)
```

### Example

```python
# Baseline latency 80 ms, target latency 90 ms, sd 40 ms
approx_n_per_arm_continuous(80, 90, sd=40, alpha=0.05, power=0.80)
```

---

## 3) Clustered-adjusted approximation

Because the generator and the downstream analysis often cluster at actor level, inflate the sample size using a simple design effect.

```python

def inflate_for_clustering(
    n_per_arm: int,
    mean_cluster_size: float,
    intra_cluster_corr: float,
) -> int:
    """
    Inflate a per-arm sample size using the standard design effect:
        DE = 1 + (m - 1) * ICC
    """
    if mean_cluster_size < 1:
        raise ValueError("mean_cluster_size must be >= 1")
    if not (0 <= intra_cluster_corr < 1):
        raise ValueError("intra_cluster_corr must be in [0, 1)")

    deff = 1 + (mean_cluster_size - 1) * intra_cluster_corr
    return math.ceil(n_per_arm * deff)
```

### Example

```python
base_n = approx_n_per_arm_binary(0.12, 0.126, alpha=0.05, power=0.80)
inflate_for_clustering(base_n, mean_cluster_size=3.5, intra_cluster_corr=0.03)
```

---

# Suggested validation scenarios to generate

A practical test suite for your A/B analysis code is:

1. **Default / balanced scenario**
   - keep the current generator unchanged

2. **Strong warm-store success**
   - make `warm_store_hit` clearly beneficial

3. **Strong session-store reuse**
   - make `session_store_hit` clearly strongest on reusable tail queries

4. **Weak fresh-compute**
   - make `fresh_compute_saved` slow and weak

5. **Inline contamination**
   - heavily route fresh-compute through `inlineSuggest`

6. **Coverage win but ranking decline**
   - raise `ITEM_COUNT`, lower clicked-rank concentration

7. **Late-period divergence**
   - create a visible shift in the last few days of the experiment

These scenarios give you a strong set of synthetic datasets for validating:
- overall ITT summaries
- popular vs non-popular slices
- mechanism-matched views
- clean-slice reruns
- daily trend analyses
- correction-burden diagnostics
- power/sample-size reasoning

---

## Final note

The important modeling principle is:

> each scenario should change the **smallest number of functions necessary** to create a clear ground-truth effect.

That makes it much easier to verify whether your downstream A/B analysis pipeline is detecting the effect you intended to simulate.
